In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 20.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    set_seed
)
import math
from torch.utils.data import DataLoader, Dataset


def compute_mlm_loss(
    csv_path,
    column_name="junction_aa",
    model_name="Chrode/H3BERTa",
    batch_size=32,
    sep=",",
    skiprows=1,
    seed=42
):

    # Seed per masking riproducibile
    set_seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # MODEL
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
    model.eval()

    # LOAD DATAFRAME
    df = pd.read_csv(csv_path, sep=sep, skiprows=skiprows)

    assert column_name in df.columns, f"Colonna '{column_name}' non trovata! Colonne: {df.columns}"

    seq_series = df[column_name].dropna().astype(str)

    # Trim: rimuove C iniziale e W finale
    seq_series = (
        seq_series
        .str.strip()
        .str.replace(r"^C", "", regex=True)
        .str.replace(r"W$", "", regex=True)
    )

    sequences = seq_series.tolist()
    print("Numero sequenze dopo trimming:", len(sequences))

    # DATASET
    class SeqDataset(Dataset):
        def __init__(self, seqs, tokenizer):
            self.seqs = seqs
            self.tokenizer = tokenizer
        def __len__(self):
            return len(self.seqs)
        def __getitem__(self, idx):
            return self.tokenizer(self.seqs[idx], truncation=True, padding=False)

    dataset = SeqDataset(sequences, tokenizer)

    # COLLATOR + DATALOADER
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15,
    )

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=data_collator,
    )

    # LOOP EVAL
    losses = []

    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            losses.append(outputs.loss.item())

    avg_loss = sum(losses) / len(losses)
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity


In [ ]:
#IgD https://www.nature.com/articles/s41586-019-0879-y. SRR8283601_1_Heavy_IGHD.csv
#{""Run"": ""SRR8283601"", ""Link"": ""https://dx.doi.org/10.1038%2Fs41586-019-0879-y"", ""Author"": ""Briney et al., 2019"", ""Species"": ""human"", ""BSource"": ""LeukoPak"", ""Vaccine"": ""None"", ""Longitudinal"": ""no"", ""Age"": ""18"", ""BType"": ""Unsorted-B-Cells"", ""Subject"": ""Subject-326650"", ""Disease"": ""None"", ""Chain"": ""Heavy"", ""Unique sequences"": 46, ""Total sequences"": 114, ""Isotype"": ""IGHD""}"

csv = "/content/drive/MyDrive/review/IgE_IgD_IgM/SRR8283601_1_Heavy_IGHD.csv"

loss, ppl = compute_mlm_loss(csv_path=csv)

print("Mean MLM loss:", loss)
print("Perplexity:", ppl)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Numero sequenze dopo trimming: 46
Mean MLM loss: 1.4521448612213135
Perplexity: 4.272268117268302


In [ ]:
#IgE https://www.cell.com/immunity/fulltext/S1074-7613(20)30504-5?_returnURL=https%3A%2F%2Flinkinghub.elsevier.com%2Fretrieve%2Fpii%2FS1074761320305045%3Fshowall%3Dtrue. Details	Bernardes_2020	53	human	IGHE	Heavy	None	None	Patient-H014	57	no
#{"Run": "SRR13082955", "Link": "https://doi.org/10.1016/j.immuni.2020.11.017", "Author": "Bernardes et al., 2020", "Species": "human", "Age": "57", "BSource": "PBMC", "BType": "Unsorted-B-Cells", "Vaccine": "None", "Disease": "None", "Subject": "Patient-H014", "Longitudinal": "no", "Unique sequences": 53, "Total sequences": 125, "Isotype": "IGHE", "Chain": "Heavy"}
csv = "/content/drive/MyDrive/review/IgE_IgD_IgM/SRR13082955_1_Heavy_IGHE.csv"

loss, ppl = compute_mlm_loss(csv_path=csv)

print("Mean MLM loss:", loss)
print("Perplexity:", ppl)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Numero sequenze dopo trimming: 53
Mean MLM loss: 1.3816527724266052
Perplexity: 3.981476667328681


In [ ]:
#IgM https://www.frontiersin.org/journals/immunology/articles/10.3389/fimmu.2019.00660/full
#{"Run": "ERR3004232", "Link": "https://doi.org/10.3389/fimmu.2019.00660", "Author": "Bernat et al., 2019", "Species": "human", "BSource": "PBMC", "BType": "Unsorted-B-Cells", "Longitudinal": "no", "Age": "no", "Disease": "None", "Subject": "A007", "Vaccine": "None", "Chain": "Heavy", "Unique sequences": 54, "Total sequences": 55, "Isotype": "IGHM"}
csv = "/content/drive/MyDrive/review/IgE_IgD_IgM/ERR3004232_1_Heavy_IGHM.csv"

loss, ppl = compute_mlm_loss(csv_path=csv)

print("Mean MLM loss:", loss)
print("Perplexity:", ppl)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Numero sequenze dopo trimming: 54
Mean MLM loss: 1.6677317023277283
Perplexity: 5.300131875019791


In [ ]:
#IgG https://genome.cshlp.org/content/23/11/1874
#{"Run": "ERR220445", "Link": "http://www.genome.org\u200b/cgi/doi/10.1101/gr.154815.113", "Author": "Bashford et al., 2013", "Species": "human", "BSource": "PBMC", "BType": "Unsorted-B-Cells", "Longitudinal": "no", "Disease": "None", "Subject": "Subject-Healthy-10", "Age": "24", "Vaccine": "None", "Chain": "Heavy", "Unique sequences": 19, "Isotype": "IGHG", "Total sequences": 20}
csv = "/content/drive/MyDrive/review/IgE_IgD_IgM/ERR220445_Heavy_IGHG.csv"

loss, ppl = compute_mlm_loss(csv_path=csv)

print("Mean MLM loss:", loss)
print("Perplexity:", ppl)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Numero sequenze dopo trimming: 19
Mean MLM loss: 1.4204692840576172
Perplexity: 4.139062380543616


In [ ]:
#IgA  https://www.nature.com/articles/s41586-019-0879-y
csv = "/content/drive/MyDrive/review/IgE_IgD_IgM/SRR8283601_1_Heavy_IGHA.csv"

loss, ppl = compute_mlm_loss(csv_path=csv)

print("Mean MLM loss:", loss)
print("Perplexity:", ppl)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/218 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/705 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Numero sequenze dopo trimming: 90
Mean MLM loss: 2.0981706778208413
Perplexity: 8.151245012215512
